<a href="https://colab.research.google.com/github/duruamobi/AAI2026/blob/main/Agentic_AI_in_Marketing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
import pandas as pd
import numpy as np

# ============================================================
# Goal: allocate daily budget across channels to maximize conversions
# Dataset expected columns:
#   date, channel, spend, impressions, clicks, conversions
# ============================================================

# -----------------------------
# 1) SYSTEM PROMPT
# -----------------------------
SYSTEM_PROMPT = """
You are an autonomous marketing budget allocation agent.

Your goal is to allocate a daily budget across channels to maximize conversions while continuing to learn from all channels.
Inputs:

-date
-channel
-spend
-impressions
-clicks
-conversions

Daily decision rules:
1. Start from the previous day’s budget allocation.
2. Use recent performance (CTR, CVR, CPA, conversions per dollar) to score channels.
3. Allocate more budget to higher-performing channels and reduce budget for lower-performing ones.
4. Apply guardrails:
-Maximum daily budget change per channel: ±20%
-Minimum budget share per channel: 15%
-No channel receives zero budget
-Total budget remains constant
5. Log each decision with a clear reason.

Objective:
-Maximize total conversions

Supporting metrics:
CTR (click-through rate)
CVR (conversion rate)
CPA (cost per acquisition)


"""

print("=" * 80)
print("SYSTEM PROMPT")
print("=" * 80)
print(SYSTEM_PROMPT.strip())
print()

# -----------------------------
# 2) CONFIG
# -----------------------------
CSV_PATH = "ad_optimization_dataset.csv"

LOOKBACK_DAYS = 3          # lightweight learning loop
MAX_DAILY_SHIFT = 0.20     # +/-20%
MIN_SHARE = 0.15           # minimum 15% of total budget to each channel
EPS = 1e-9

# -----------------------------
# 3) LOAD + VALIDATE DATA
# -----------------------------
def load_data(csv_path: str) -> pd.DataFrame:
    """Read CSV and validate required columns."""
    df = pd.read_csv(csv_path)

    required_cols = {"date", "channel", "spend", "impressions", "clicks", "conversions"}
    missing = required_cols - set(df.columns)
    if missing:
        raise ValueError(f"Missing required columns: {missing}")

    # Parse date column
    df["date"] = pd.to_datetime(df["date"])

    # Aggregate just in case there are multiple rows per date-channel
    df = (
        df.groupby(["date", "channel"], as_index=False)
          .agg({
              "spend": "sum",
              "impressions": "sum",
              "clicks": "sum",
              "conversions": "sum"
          })
          .sort_values(["date", "channel"])
    )

    return df

# -----------------------------
# 4) METRIC CALCULATIONS
# -----------------------------
def add_metrics(df: pd.DataFrame) -> pd.DataFrame:
    """Calculate performance metrics."""
    out = df.copy()
    out["ctr"] = out["clicks"] / (out["impressions"] + EPS)
    out["cvr"] = out["conversions"] / (out["clicks"] + EPS)
    out["cpc"] = out["spend"] / (out["clicks"] + EPS)
    out["cpa"] = out["spend"] / (out["conversions"] + EPS)
    out["conv_per_dollar"] = out["conversions"] / (out["spend"] + EPS)
    return out

# -----------------------------
# 5) HELPER FUNCTIONS
# -----------------------------
def normalize(series: pd.Series) -> pd.Series:
    """Convert a positive series to shares that sum to 1."""
    total = series.sum()
    if total <= 0:
        return pd.Series(np.ones(len(series)) / len(series), index=series.index)
    return series / total

def apply_min_share(shares: pd.Series, min_share: float) -> pd.Series:
    """
    Enforce a minimum share for each channel, then renormalize.
    """
    shares = shares.copy()

    # Simple iterative approach
    for _ in range(10):
        low = shares < min_share
        if not low.any():
            break

        deficit = (min_share - shares[low]).sum()
        shares[low] = min_share

        high = ~low
        if high.any():
            high_total = shares[high].sum()
            if high_total > 0:
                shares[high] = shares[high] * max(high_total - deficit, 0) / high_total

        shares = shares / shares.sum()

    return shares

def cap_daily_change(prev_shares: pd.Series, new_shares: pd.Series, max_shift: float) -> pd.Series:
    """
    Cap per-channel daily movement to +/- max_shift relative to yesterday's share.
    """
    capped = new_shares.copy()

    for ch in capped.index:
        lower = prev_shares[ch] * (1 - max_shift)
        upper = prev_shares[ch] * (1 + max_shift)
        capped[ch] = min(max(capped[ch], lower), upper)

    # Renormalize after capping
    capped = capped / capped.sum()
    return capped

def score_channels(history: pd.DataFrame) -> pd.DataFrame:
    """
    Score channels using recent performance.
    Heavier weight on conversion efficiency, with CVR and CTR as supporting signals.
    """
    hist = (
        history.groupby("channel", as_index=False)
               .agg({
                   "spend": "sum",
                   "impressions": "sum",
                   "clicks": "sum",
                   "conversions": "sum"
               })
    )

    hist["ctr"] = hist["clicks"] / (hist["impressions"] + EPS)
    hist["cvr"] = hist["conversions"] / (hist["clicks"] + EPS)
    hist["cpa"] = hist["spend"] / (hist["conversions"] + EPS)
    hist["conv_per_dollar"] = hist["conversions"] / (hist["spend"] + EPS)

    # Blended score
    hist["score"] = (
        0.60 * normalize(hist["conv_per_dollar"]) +
        0.30 * normalize(hist["cvr"]) +
        0.10 * normalize(hist["ctr"])
    )

    return hist

# -----------------------------
# 6) BASELINE + AGENT SIMULATION
# -----------------------------
def run_simulation(df: pd.DataFrame):
    """
    Compare:
    - Baseline: equal split each day
    - Agent: dynamic allocation based on recent performance

    Offline evaluation assumption:
    same-day channel efficiency (conversions per dollar) stays constant
    under reallocated budget.
    """
    data = add_metrics(df)

    channels = sorted(data["channel"].unique())
    dates = sorted(data["date"].unique())

    # Total daily budget comes from observed dataset spend
    daily_total_budget = data.groupby("date")["spend"].sum().to_dict()

    # Start with equal shares
    prev_agent_shares = pd.Series(1 / len(channels), index=channels)

    daily_logs = []
    allocation_rows = []

    for day_idx, current_date in enumerate(dates):
        current_day = data[data["date"] == current_date].set_index("channel").loc[channels]
        total_budget = daily_total_budget[current_date]

        # Baseline: equal split
        baseline_shares = pd.Series(1 / len(channels), index=channels)
        baseline_budget = baseline_shares * total_budget

        # Agent logic
        if day_idx < LOOKBACK_DAYS:
            agent_shares = baseline_shares.copy()
            reason = f"Used equal split because only {day_idx} prior days of history were available."
            score_table = pd.DataFrame({"channel": channels, "score": [np.nan] * len(channels)}).set_index("channel")
        else:
            history_dates = dates[day_idx - LOOKBACK_DAYS : day_idx]
            history = data[data["date"].isin(history_dates)]

            score_table = score_channels(history).set_index("channel").loc[channels]

            # Proposed shares from score
            proposed_shares = normalize(score_table["score"])
            proposed_shares = apply_min_share(proposed_shares, MIN_SHARE)

            # Cap daily shift
            capped_shares = cap_daily_change(prev_agent_shares, proposed_shares, MAX_DAILY_SHIFT)
            capped_shares = apply_min_share(capped_shares, MIN_SHARE)

            agent_shares = capped_shares.copy()

            # Build readable reason string
            top_channel = score_table["score"].idxmax()
            low_channel = score_table["score"].idxmin()
            top_cvr = score_table.loc[top_channel, "cvr"]
            low_cpa = score_table.loc[top_channel, "cpa"]

            reason = (
                f"Increased {top_channel} share because it had the strongest recent blended score "
                f"(high conversion efficiency/CVR, recent CVR={top_cvr:.3f}, CPA={low_cpa:.2f}). "
                f"Reduced relative share for {low_channel}. "
                f"Applied guardrails: max daily shift={int(MAX_DAILY_SHIFT*100)}%, "
                f"minimum share={int(MIN_SHARE*100)}%."
            )

        agent_budget = agent_shares * total_budget

        # Offline evaluation using same-day observed efficiency
        current_day["conv_per_dollar"] = current_day["conversions"] / (current_day["spend"] + EPS)
        current_day["clicks_per_dollar"] = current_day["clicks"] / (current_day["spend"] + EPS)

        baseline_est_conversions = float((baseline_budget * current_day["conv_per_dollar"]).sum())
        agent_est_conversions = float((agent_budget * current_day["conv_per_dollar"]).sum())

        baseline_est_clicks = float((baseline_budget * current_day["clicks_per_dollar"]).sum())
        agent_est_clicks = float((agent_budget * current_day["clicks_per_dollar"]).sum())

        baseline_cpa = total_budget / (baseline_est_conversions + EPS)
        agent_cpa = total_budget / (agent_est_conversions + EPS)

        daily_logs.append({
            "date": current_date,
            "reason": reason,
            "baseline_est_conversions": baseline_est_conversions,
            "agent_est_conversions": agent_est_conversions,
            "baseline_est_clicks": baseline_est_clicks,
            "agent_est_clicks": agent_est_clicks,
            "baseline_cpa": baseline_cpa,
            "agent_cpa": agent_cpa
        })

        for ch in channels:
            allocation_rows.append({
                "date": current_date,
                "channel": ch,
                "actual_spend": current_day.loc[ch, "spend"],
                "actual_impressions": current_day.loc[ch, "impressions"],
                "actual_clicks": current_day.loc[ch, "clicks"],
                "actual_conversions": current_day.loc[ch, "conversions"],
                "ctr": current_day.loc[ch, "ctr"],
                "cvr": current_day.loc[ch, "cvr"],
                "cpa": current_day.loc[ch, "cpa"],
                "baseline_budget": baseline_budget[ch],
                "agent_budget": agent_budget[ch],
                "agent_share": agent_shares[ch],
                "score_used": score_table.loc[ch, "score"] if ch in score_table.index else np.nan,
                "decision_reason": reason
            })

        prev_agent_shares = agent_shares.copy()

    allocation_df = pd.DataFrame(allocation_rows)
    log_df = pd.DataFrame(daily_logs)

    return data, allocation_df, log_df

# -----------------------------
# 7) EVALUATION SUMMARY
# -----------------------------
def summarize_results(log_df: pd.DataFrame):
    total_baseline_conversions = log_df["baseline_est_conversions"].sum()
    total_agent_conversions = log_df["agent_est_conversions"].sum()

    total_baseline_clicks = log_df["baseline_est_clicks"].sum()
    total_agent_clicks = log_df["agent_est_clicks"].sum()

    # total spend is same for both methods in this setup
    total_spend = (log_df["baseline_cpa"] * log_df["baseline_est_conversions"]).sum()

    baseline_avg_cpa = total_spend / (total_baseline_conversions + EPS)
    agent_avg_cpa = total_spend / (total_agent_conversions + EPS)

    conv_lift_pct = ((total_agent_conversions - total_baseline_conversions) / (total_baseline_conversions + EPS)) * 100
    click_lift_pct = ((total_agent_clicks - total_baseline_clicks) / (total_baseline_clicks + EPS)) * 100

    print("=" * 80)
    print("EVALUATION")
    print("=" * 80)
    print(f"Baseline estimated total conversions: {total_baseline_conversions:.2f}")
    print(f"Agent estimated total conversions:    {total_agent_conversions:.2f}")
    print(f"Conversion lift vs baseline:          {conv_lift_pct:.2f}%")
    print()
    print(f"Baseline estimated total clicks:      {total_baseline_clicks:.2f}")
    print(f"Agent estimated total clicks:         {total_agent_clicks:.2f}")
    print(f"Click lift vs baseline:               {click_lift_pct:.2f}%")
    print()
    print(f"Baseline average CPA:                 {baseline_avg_cpa:.2f}")
    print(f"Agent average CPA:                    {agent_avg_cpa:.2f}")
    print()

# -----------------------------
# 8) MAIN
# -----------------------------
def main():
    df = load_data(CSV_PATH)

    print("=" * 80)
    print("DATA CHECK")
    print("=" * 80)
    print(df.head())
    print()
    print("Rows:", len(df))
    print("Channels:", df["channel"].unique())
    print("Unique days:", df["date"].nunique())
    print()

    metrics_df, allocation_df, log_df = run_simulation(df)

    # Print metric sample
    print("=" * 80)
    print("METRICS SAMPLE")
    print("=" * 80)
    print(metrics_df.head(10).to_string(index=False))
    print()

    # Print some decision logs
    print("=" * 80)
    print("DECISION LOG SAMPLE")
    print("=" * 80)
    print(log_df[["date", "reason", "baseline_est_conversions", "agent_est_conversions", "baseline_cpa", "agent_cpa"]]
          .head(10)
          .to_string(index=False))
    print()

    summarize_results(log_df)

    # Save outputs for report/presentation
    allocation_df.to_csv("agent_allocation_output.csv", index=False)
    log_df.to_csv("agent_decision_log.csv", index=False)

    print("Saved files:")
    print("- agent_allocation_output.csv")
    print("- agent_decision_log.csv")

if __name__ == "__main__":
    main()

SYSTEM PROMPT
You are an autonomous marketing budget allocation agent.

Your goal is to allocate a daily budget across channels to maximize conversions while continuing to learn from all channels.
Inputs:

-date
-channel
-spend
-impressions
-clicks
-conversions

Daily decision rules:
1. Start from the previous day’s budget allocation.
2. Use recent performance (CTR, CVR, CPA, conversions per dollar) to score channels.
3. Allocate more budget to higher-performing channels and reduce budget for lower-performing ones.
4. Apply guardrails:
-Maximum daily budget change per channel: ±20%
-Minimum budget share per channel: 15%
-No channel receives zero budget
-Total budget remains constant
5. Log each decision with a clear reason.

Objective:
-Maximize total conversions

Supporting metrics:
CTR (click-through rate)
CVR (conversion rate)
CPA (cost per acquisition)

DATA CHECK
        date   channel   spend  impressions  clicks  conversions
0 2025-01-01     Email  150.77        64131    1832   